# LAP × 联想记忆：容量与干预鲁棒性

这个 notebook 会完成链式 SCM 与混杂 fork 的数据生成、Modern Hopfield 对照、LAP-regularized E-SCM 训练、do-intervention、容量/吸引域评估、配对统计和错误 DAG 消融。

> 默认配置是端到端烟雾实验。按 `smoke → confirmatory → full` 递进运行；CSV 支持断点续跑，正式结论只使用 `full`。

## 1. 自包含实验代码

数据生成、Modern Hopfield、E-SCM、LAP 二阶导、三步干预、评估与绘图均在下面一个代码单元格中，不需要克隆仓库或导入伴随 `.py` 文件。每个局部机制 $E_i$ 使用固定的 Modern Hopfield 能量存储 $(z_i,z_{PA(i)})$ 元组；full-state residual 刻意保留非法耦合通道，再由 LAP 约束。额外的 hard-mask local Hopfield 关闭 residual，作为架构直接保证模块化时的性能上界。

In [ ]:
#@title 完整实验实现（自包含，运行本单元格）
"""End-to-end LAP associative-memory experiment.

The important modelling choice is that every mechanism has two branches:

1. a fixed local Modern Hopfield branch that stores each node-parent tuple;
2. a weak full-state residual branch that can create illegal coupling.

Without the second branch, the LAP mixed partial is identically zero by
construction, so changing ``lambda_lap`` would not test anything. LAP is used to
regularize the residual branch while the local Hopfield branch supplies attractors.
"""

from __future__ import annotations

import copy
import itertools
import json
import math
import platform
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Iterable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import Tensor, nn
from torch.nn import functional as F

EXPERIMENT_SCHEMA_VERSION = 2


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def bipolar(x: Tensor) -> Tensor:
    """Return {-1,+1} values, mapping exact zero to +1."""

    return torch.where(x >= 0, torch.ones_like(x), -torch.ones_like(x))


def flip_bits(x: Tensor, probability: float, *, generator: torch.Generator | None = None) -> Tensor:
    if not 0 <= probability <= 1:
        raise ValueError("probability must be in [0, 1]")
    mask = torch.rand(x.shape, device=x.device, generator=generator) < probability
    return torch.where(mask, -x, x)


@dataclass(frozen=True)
class SCMData:
    graph: str
    patterns: Tensor
    nodes: tuple[str, ...]
    parents: Mapping[str, tuple[str, ...]]
    node_dim: int
    weights: Mapping[str, Tensor]

    @property
    def total_dim(self) -> int:
        return len(self.nodes) * self.node_dim

    @property
    def slices(self) -> dict[str, slice]:
        return {
            node: slice(i * self.node_dim, (i + 1) * self.node_dim)
            for i, node in enumerate(self.nodes)
        }


def _random_weight(dim: int, generator: torch.Generator) -> Tensor:
    # 1/sqrt(d) keeps pre-activations well scaled; signs remain deterministic.
    signs = torch.randint(0, 2, (dim, dim), generator=generator).float().mul_(2).sub_(1)
    return signs / math.sqrt(dim)


def generate_chain(
    n_patterns: int,
    node_dim: int = 10,
    p_flip: float = 0.05,
    seed: int = 0,
) -> SCMData:
    """Generate X -> Y -> Z observational patterns."""

    generator = torch.Generator().manual_seed(seed)
    w_xy = _random_weight(node_dim, generator)
    w_yz = _random_weight(node_dim, generator)
    x = bipolar(torch.randn(n_patterns, node_dim, generator=generator))
    y_clean = bipolar(x @ w_xy.T)
    y = flip_bits(y_clean, p_flip, generator=generator)
    z_clean = bipolar(y @ w_yz.T)
    z = flip_bits(z_clean, p_flip, generator=generator)
    return SCMData(
        graph="chain",
        patterns=torch.cat((x, y, z), dim=1),
        nodes=("X", "Y", "Z"),
        parents={"X": (), "Y": ("X",), "Z": ("Y",)},
        node_dim=node_dim,
        weights={"XY": w_xy, "YZ": w_yz},
    )


def generate_fork(
    n_patterns: int,
    node_dim: int = 10,
    p_flip: float = 0.05,
    seed: int = 0,
) -> SCMData:
    """Generate the confounded fork C -> X and C -> Y."""

    generator = torch.Generator().manual_seed(seed)
    w_cx = _random_weight(node_dim, generator)
    w_cy = _random_weight(node_dim, generator)
    c = bipolar(torch.randn(n_patterns, node_dim, generator=generator))
    x = flip_bits(bipolar(c @ w_cx.T), p_flip, generator=generator)
    y = flip_bits(bipolar(c @ w_cy.T), p_flip, generator=generator)
    return SCMData(
        graph="fork",
        patterns=torch.cat((c, x, y), dim=1),
        nodes=("C", "X", "Y"),
        parents={"C": (), "X": ("C",), "Y": ("C",)},
        node_dim=node_dim,
        weights={"CX": w_cx, "CY": w_cy},
    )


def reverse_parent_map(
    nodes: Sequence[str], parents: Mapping[str, Sequence[str]]
) -> dict[str, tuple[str, ...]]:
    """Reverse every declared edge while preserving the node set."""

    reversed_parents: dict[str, list[str]] = {node: [] for node in nodes}
    for child in nodes:
        for parent in parents[child]:
            reversed_parents[parent].append(child)
    return {node: tuple(reversed_parents[node]) for node in nodes}


class EnergyMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x).squeeze(-1)


class LocalModernHopfieldEnergy(nn.Module):
    """Fixed Modern Hopfield energy over one node and its declared parents.

    The quadratic term is essential: without it, stored tuples are not fixed
    points of gradient descent. Pattern-norm biases make the expression valid
    beyond equal-norm bipolar memories.
    """

    def __init__(self, patterns: Tensor, beta: float = 4.0):
        super().__init__()
        if patterns.ndim != 2 or len(patterns) == 0:
            raise ValueError("patterns must be a non-empty [N, D] tensor")
        self.beta = beta
        self.register_buffer("patterns", patterns.detach().clone())
        self.register_buffer("half_pattern_norms", 0.5 * patterns.detach().square().sum(dim=-1))

    def forward(self, x: Tensor) -> Tensor:
        logits = self.beta * (x @ self.patterns.T - self.half_pattern_norms)
        return 0.5 * x.square().sum(dim=-1) - torch.logsumexp(logits, dim=-1) / self.beta


class CausalEnergyMemory(nn.Module):
    """Local Hopfield attractors plus LAP-regularized illegal residuals."""

    def __init__(
        self,
        nodes: Sequence[str],
        memory_parents: Mapping[str, Sequence[str]],
        patterns: Tensor,
        node_dim: int,
        hidden_dim: int = 64,
        residual_scale: float = 0.25,
        hopfield_beta: float = 4.0,
        lap_parents: Mapping[str, Sequence[str]] | None = None,
    ):
        super().__init__()
        self.nodes = tuple(nodes)
        self.memory_parents = {node: tuple(memory_parents[node]) for node in self.nodes}
        active_lap_parents = memory_parents if lap_parents is None else lap_parents
        self.lap_parents = {node: tuple(active_lap_parents[node]) for node in self.nodes}
        self.node_dim = node_dim
        self.total_dim = len(self.nodes) * node_dim
        self.residual_scale = residual_scale
        self.slices = {
            node: slice(i * node_dim, (i + 1) * node_dim)
            for i, node in enumerate(self.nodes)
        }
        self.legal = nn.ModuleDict()
        self.residual = nn.ModuleDict()
        for node in self.nodes:
            local_patterns = self._legal_input(patterns, node)
            self.legal[node] = LocalModernHopfieldEnergy(local_patterns, beta=hopfield_beta)
            self.residual[node] = EnergyMLP(self.total_dim, hidden_dim)

    def _legal_input(self, z: Tensor, node: str) -> Tensor:
        ordered = (node,) + self.memory_parents[node]
        return torch.cat([z[..., self.slices[name]] for name in ordered], dim=-1)

    def mechanism_energies(self, z: Tensor) -> dict[str, Tensor]:
        return {
            node: self.legal[node](self._legal_input(z, node))
            + self.residual_scale * self.residual[node](z)
            for node in self.nodes
        }

    def forward(self, z: Tensor) -> Tensor:
        energies = self.mechanism_energies(z)
        return torch.stack(list(energies.values()), dim=0).sum(dim=0)

    def lap_penalty(
        self,
        z: Tensor,
        probes: int = 1,
        parents: Mapping[str, Sequence[str]] | None = None,
        generator: torch.Generator | None = None,
    ) -> Tensor:
        """Hutchinson estimate of illegal mixed-Hessian Frobenius norms.

        For every mechanism i, this estimates
        ||d^2 E_i / (d z_i d z_A)||_F^2 for nodes A that are neither i nor a
        parent of i. Two calls to ``autograd.grad`` compute each mixed partial.
        """

        if probes < 1:
            raise ValueError("probes must be >= 1")
        if not z.requires_grad:
            z = z.detach().requires_grad_(True)
        active_parents = self.lap_parents if parents is None else {
            node: tuple(parents[node]) for node in self.nodes
        }

        penalties: list[Tensor] = []
        energies = self.mechanism_energies(z)
        for node, energy in energies.items():
            grad_all = torch.autograd.grad(
                energy.sum(), z, create_graph=True, retain_graph=True
            )[0]
            grad_self = grad_all[..., self.slices[node]]
            illegal_nodes = [
                other
                for other in self.nodes
                if other != node and other not in active_parents[node]
            ]
            for _ in range(probes):
                probe = torch.empty_like(grad_self).bernoulli_(0.5, generator=generator).mul_(2).sub_(1)
                mixed_all = torch.autograd.grad(
                    (grad_self * probe).sum(),
                    z,
                    create_graph=True,
                    retain_graph=True,
                )[0]
                for illegal in illegal_nodes:
                    penalties.append(mixed_all[..., self.slices[illegal]].square().mean())

        if not penalties:
            return z.sum() * 0.0
        return torch.stack(penalties).mean()


class ModernHopfield:
    """Non-parametric log-sum-exp Modern Hopfield baseline."""

    def __init__(self, patterns: Tensor, beta: float = 1.0):
        self.patterns = patterns.detach().clone()
        self.beta = beta

    def energy(self, z: Tensor) -> Tensor:
        similarities = self.beta * (z @ self.patterns.T - 0.5 * self.patterns.square().sum(dim=-1))
        return 0.5 * z.square().sum(dim=-1) - torch.logsumexp(similarities, dim=-1) / self.beta

    def retrieve(
        self,
        initial: Tensor,
        clamp_mask: Tensor | None = None,
        clamp_values: Tensor | None = None,
        steps: int = 50,
        damping: float = 0.8,
        tolerance: float = 1e-5,
        discretize: bool = True,
    ) -> Tensor:
        z = initial.detach().clone()
        if clamp_mask is None:
            clamp_mask = torch.zeros_like(z, dtype=torch.bool)
        if clamp_values is None:
            clamp_values = z
        for _ in range(steps):
            weights = torch.softmax(self.beta * (z @ self.patterns.T), dim=-1)
            proposal = weights @ self.patterns
            updated = (1 - damping) * z + damping * proposal
            updated = torch.where(clamp_mask, clamp_values, updated)
            if (updated - z).abs().max().item() < tolerance:
                z = updated
                break
            z = updated
        return bipolar(z) if discretize else z


@dataclass(frozen=True)
class TrainingConfig:
    epochs: int = 120
    batch_size: int = 16
    learning_rate: float = 2e-3
    negative_flip: float = 0.30
    negatives_per_positive: int = 2
    margin: float = 1.0
    hidden_dim: int = 48
    residual_scale: float = 0.10
    hopfield_beta: float = 4.0
    denoise_flip: float = 0.15
    stationarity_weight: float = 1.0
    denoise_weight: float = 1.0
    lap_probes: int = 1
    log_every: int = 10


@dataclass(frozen=True)
class RetrievalConfig:
    steps: int = 80
    learning_rate: float = 0.12
    tolerance: float = 1e-4


@dataclass(frozen=True)
class ExperimentConfig:
    node_dim: int = 6
    pattern_counts: tuple[int, ...] = (4, 8)
    lambdas: tuple[float, ...] = (0.0, 1e2, 1e3, 1e4, 1e5)
    seeds: tuple[int, ...] = (0,)
    p_flip: float = 0.05
    capacity_noise: float = 0.10
    basin_noise: tuple[float, ...] = (0.0, 0.2, 0.4)
    training: TrainingConfig = TrainingConfig()
    retrieval: RetrievalConfig = RetrievalConfig()

    @classmethod
    def confirmatory(cls) -> "ExperimentConfig":
        """Five-seed stability check with a manageable Colab runtime."""

        return cls(
            node_dim=10,
            pattern_counts=(5, 10, 20, 50),
            lambdas=(0.0, 1e2, 1e3, 1e4, 1e5),
            seeds=(0, 1, 2, 3, 4),
            basin_noise=(0.0, 0.1, 0.2, 0.3, 0.4),
            training=TrainingConfig(epochs=200, batch_size=32, hidden_dim=64),
            retrieval=RetrievalConfig(steps=120, learning_rate=0.08),
        )

    @classmethod
    def full(cls) -> "ExperimentConfig":
        return cls(
            node_dim=10,
            pattern_counts=(5, 10, 20, 50, 100),
            lambdas=(0.0, 1e2, 1e3, 1e4, 1e5),
            seeds=tuple(range(10)),
            basin_noise=(0.0, 0.1, 0.2, 0.3, 0.4, 0.5),
            training=TrainingConfig(epochs=300, batch_size=32, hidden_dim=64),
            retrieval=RetrievalConfig(steps=150, learning_rate=0.08),
        )


def train_energy_memory(
    data: SCMData,
    lambda_lap: float,
    config: TrainingConfig,
    seed: int,
    device: torch.device,
    lap_parents: Mapping[str, Sequence[str]] | None = None,
) -> tuple[CausalEnergyMemory, pd.DataFrame]:
    set_seed(seed)
    model = CausalEnergyMemory(
        data.nodes,
        data.parents,
        data.patterns,
        data.node_dim,
        hidden_dim=config.hidden_dim,
        residual_scale=config.residual_scale,
        hopfield_beta=config.hopfield_beta,
        lap_parents=lap_parents,
    ).to(device)
    patterns = data.patterns.to(device)
    data_generator = torch.Generator(device=device).manual_seed(seed + 1_000_003)
    lap_generator = torch.Generator(device=device).manual_seed(seed + 2_000_003)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    history: list[dict[str, float | int]] = []

    for epoch in range(config.epochs):
        permutation = torch.randperm(len(patterns), device=device, generator=data_generator)
        epoch_recon = 0.0
        epoch_stationarity = 0.0
        epoch_denoise = 0.0
        batches = 0
        for start in range(0, len(patterns), config.batch_size):
            positive = patterns[permutation[start : start + config.batch_size]]
            positive = positive.detach().requires_grad_(True)
            negatives = [
                flip_bits(positive.detach(), config.negative_flip, generator=data_generator)
                for _ in range(config.negatives_per_positive)
            ]
            negative = torch.cat(negatives, dim=0)
            noisy = flip_bits(positive.detach(), config.denoise_flip, generator=data_generator).requires_grad_(True)

            e_pos = model(positive)
            e_neg = model(negative).reshape(config.negatives_per_positive, len(positive)).mean(dim=0)
            recon = F.softplus(config.margin + e_pos - e_neg).mean()
            positive_gradient = torch.autograd.grad(
                e_pos.sum(), positive, create_graph=True, retain_graph=True
            )[0]
            stationarity = positive_gradient.square().mean()
            noisy_gradient = torch.autograd.grad(
                model(noisy).sum(), noisy, create_graph=True, retain_graph=True
            )[0]
            # For a quadratic well centered at the clean pattern, grad E(noisy)=noisy-clean.
            target_gradient = noisy.detach() - positive.detach()
            denoise = F.mse_loss(noisy_gradient, target_gradient)
            lap = model.lap_penalty(positive, probes=config.lap_probes, generator=lap_generator) if lambda_lap > 0 else recon.new_zeros(())
            task_loss = (
                recon
                + config.stationarity_weight * stationarity
                + config.denoise_weight * denoise
            )
            loss = task_loss + lambda_lap * lap

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            epoch_recon += float(recon.detach())
            epoch_stationarity += float(stationarity.detach())
            epoch_denoise += float(denoise.detach())
            batches += 1

        should_log = epoch == 0 or (epoch + 1) % config.log_every == 0 or epoch + 1 == config.epochs
        if should_log:
            # Every lambda uses the same held-out states and probe seed for comparable diagnostics.
            diagnostic_batch = patterns[: min(config.batch_size, len(patterns))].detach().requires_grad_(True)
            diagnostic_generator = torch.Generator(device=device).manual_seed(seed + 9_000_000 + epoch)
            diagnostic_lap = float(
                model.lap_penalty(diagnostic_batch, probes=config.lap_probes, generator=diagnostic_generator).detach()
            )
            history.append(
                {
                    "epoch": epoch + 1,
                    "recon_loss": epoch_recon / max(batches, 1),
                    "stationarity_loss": epoch_stationarity / max(batches, 1),
                    "denoise_loss": epoch_denoise / max(batches, 1),
                    "lap_loss": diagnostic_lap,
                    "task_loss": (
                        epoch_recon / max(batches, 1)
                        + config.stationarity_weight * epoch_stationarity / max(batches, 1)
                        + config.denoise_weight * epoch_denoise / max(batches, 1)
                    ),
                    "weighted_lap_loss": lambda_lap * diagnostic_lap,
                    "total_loss": (
                        epoch_recon / max(batches, 1)
                        + config.stationarity_weight * epoch_stationarity / max(batches, 1)
                        + config.denoise_weight * epoch_denoise / max(batches, 1)
                        + lambda_lap * diagnostic_lap
                    ),
                }
            )
    return model, pd.DataFrame(history)


def relax_energy(
    model: CausalEnergyMemory,
    initial: Tensor,
    config: RetrievalConfig,
    clamp_mask: Tensor | None = None,
    clamp_values: Tensor | None = None,
    discretize: bool = True,
) -> Tensor:
    z = initial.detach().clone()
    if clamp_mask is None:
        clamp_mask = torch.zeros_like(z, dtype=torch.bool)
    if clamp_values is None:
        clamp_values = z
    for _ in range(config.steps):
        z = z.detach().requires_grad_(True)
        gradient = torch.autograd.grad(model(z).sum(), z)[0]
        free_gradient = gradient.masked_fill(clamp_mask, 0.0)
        updated = (z - config.learning_rate * free_gradient).clamp(-1.0, 1.0)
        updated = torch.where(clamp_mask, clamp_values, updated).detach()
        if free_gradient.abs().max().item() < config.tolerance:
            z = updated
            break
        z = updated
    return bipolar(z) if discretize else z


Retriever = Callable[[Tensor, Tensor | None, Tensor | None], Tensor]


def _energy_retriever(
    model: CausalEnergyMemory, config: RetrievalConfig, *, discretize: bool = True
) -> Retriever:
    return lambda initial, mask=None, values=None: relax_energy(
        model, initial, config, mask, values, discretize=discretize
    )


def _hopfield_retriever(model: ModernHopfield, *, discretize: bool = True) -> Retriever:
    return lambda initial, mask=None, values=None: model.retrieve(
        initial, mask, values, discretize=discretize
    )


def abduct_intervene_predict(
    retrieve: Retriever,
    observation: Tensor,
    observed_mask: Tensor,
    intervention_mask: Tensor,
    intervention_values: Tensor,
    preserve_mask: Tensor | None = None,
) -> tuple[Tensor, Tensor]:
    """Run abduction, hard intervention, and counterfactual prediction.

    ``observed_mask`` is clamped during abduction. Prediction starts from the
    abduced state, replaces the intervened coordinates, and only clamps the
    intervention plus an optional factual context (for example the confounder C).
    """

    abduced = retrieve(observation, observed_mask, observation)
    counterfactual_start = torch.where(intervention_mask, intervention_values, abduced)
    prediction_mask = intervention_mask.clone()
    if preserve_mask is not None:
        prediction_mask = prediction_mask | preserve_mask
    predicted = retrieve(counterfactual_start, prediction_mask, counterfactual_start)
    return abduced, predicted


def retrieval_scores(
    retrieve: Retriever, patterns: Tensor, noise: float, *, seed: int | None = None
) -> dict[str, float]:
    generator = None if seed is None else torch.Generator(device=patterns.device).manual_seed(seed)
    corrupted = flip_bits(patterns, noise, generator=generator)
    recalled = retrieve(corrupted, None, None)
    match = recalled.eq(patterns)
    return {
        "exact_retrieval": float(match.all(dim=-1).float().mean()),
        "bit_accuracy": float(match.float().mean()),
    }


def fork_leakage(retrieve_continuous: Retriever, data: SCMData) -> dict[str, float]:
    """Matched counterfactual effect of X on Y with C fixed.

    Both factual-control and do(X) branches release Y for the same relaxation
    steps. Their difference therefore excludes ordinary off-attractor drift.
    """

    slices = data.slices
    observation = data.patterns.clone()
    do_values = observation.clone()
    do_values[:, slices["X"]] *= -1
    observed_mask = torch.ones_like(observation, dtype=torch.bool)
    intervention_mask = torch.zeros_like(observation, dtype=torch.bool)
    intervention_mask[:, slices["X"]] = True
    preserve_mask = torch.zeros_like(observation, dtype=torch.bool)
    preserve_mask[:, slices["C"]] = True
    abduced = retrieve_continuous(observation, observed_mask, observation)
    prediction_mask = intervention_mask | preserve_mask
    factual_control = retrieve_continuous(abduced, prediction_mask, abduced)
    counterfactual_start = torch.where(intervention_mask, do_values, abduced)
    counterfactual = retrieve_continuous(
        counterfactual_start, prediction_mask, counterfactual_start
    )
    before_y = factual_control[:, slices["Y"]]
    after_y = counterfactual[:, slices["Y"]]
    # RMS distance divided by the bipolar range gives a normalized continuous leakage.
    normalized_l2 = (after_y - before_y).square().mean(dim=-1).sqrt() / 2
    control_drift = (before_y - abduced[:, slices["Y"]]).square().mean(dim=-1).sqrt() / 2
    after_y_bits = bipolar(after_y)
    return {
        "leakage_continuous": float(normalized_l2.mean()),
        "leakage_bit_fraction": float(after_y_bits.ne(bipolar(before_y)).float().mean()),
        "control_relaxation_drift": float(control_drift.mean()),
    }


def chain_causal_accuracy(retrieve: Retriever, data: SCMData) -> dict[str, float]:
    """Swap in a stored root X and test whether its stored descendants follow.

    This is an in-support intervention: it tests causal recombination without
    confounding it with out-of-distribution function approximation.
    """

    slices = data.slices
    observation = data.patterns.clone()
    donor = observation.roll(shifts=1, dims=0)
    do_values = observation.clone()
    do_values[:, slices["X"]] = donor[:, slices["X"]]
    observed_mask = torch.ones_like(observation, dtype=torch.bool)
    intervention_mask = torch.zeros_like(observation, dtype=torch.bool)
    intervention_mask[:, slices["X"]] = True
    _, after = abduct_intervene_predict(
        retrieve,
        observation,
        observed_mask,
        intervention_mask,
        do_values,
    )
    expected_y = donor[:, slices["Y"]]
    expected_z = donor[:, slices["Z"]]
    y_acc = after[:, slices["Y"]].eq(expected_y).float().mean()
    z_acc = after[:, slices["Z"]].eq(expected_z).float().mean()
    return {"causal_accuracy_y": float(y_acc), "causal_accuracy_z": float(z_acc)}


def _metric_rows(
    values: Mapping[str, float],
    *,
    model: str,
    lambda_lap: float | None,
    n_patterns: int,
    seed: int,
    graph: str,
    noise: float | None = None,
    evaluation: str,
) -> list[dict[str, object]]:
    return [
        {
            "model": model,
            "lambda_lap": lambda_lap,
            "n_patterns": n_patterns,
            "seed": seed,
            "graph": graph,
            "evaluation": evaluation,
            "noise": noise,
            "metric": metric,
            "value": value,
        }
        for metric, value in values.items()
    ]


def _evaluate_model(
    retrieve: Retriever,
    chain: SCMData,
    fork: SCMData,
    config: ExperimentConfig,
    *,
    model_name: str,
    lambda_lap: float | None,
    seed: int,
) -> list[dict[str, object]]:
    n_patterns = len(chain.patterns)
    def evaluation_seed(noise: float) -> int:
        return seed * 10_000_000 + n_patterns * 10_000 + int(round(noise * 1_000))

    rows = _metric_rows(
        retrieval_scores(
            retrieve, chain.patterns, config.capacity_noise, seed=evaluation_seed(config.capacity_noise)
        ),
        model=model_name,
        lambda_lap=lambda_lap,
        n_patterns=n_patterns,
        seed=seed,
        graph="chain",
        noise=config.capacity_noise,
        evaluation="capacity",
    )
    for noise in config.basin_noise:
        rows.extend(
            _metric_rows(
                retrieval_scores(retrieve, chain.patterns, noise, seed=evaluation_seed(noise)),
                model=model_name,
                lambda_lap=lambda_lap,
                n_patterns=n_patterns,
                seed=seed,
                graph="chain",
                noise=noise,
                evaluation="basin",
            )
        )
    rows.extend(
        _metric_rows(
            chain_causal_accuracy(retrieve, chain),
            model=model_name,
            lambda_lap=lambda_lap,
            n_patterns=n_patterns,
            seed=seed,
            graph="chain",
            evaluation="causal_intervention",
        )
    )
    # Leakage must use a model trained on fork data, so it is appended separately.
    return rows


def lap_structure_diagnostics(
    model: CausalEnergyMemory,
    patterns: Tensor,
    true_parents: Mapping[str, Sequence[str]],
    *,
    probes: int = 4,
    seed: int = 0,
) -> dict[str, float]:
    """Measure the penalty under both the declared and ground-truth masks."""

    z = patterns.detach().requires_grad_(True)
    values: dict[str, float] = {}
    for label, parents in (("declared", model.lap_parents), ("true", true_parents)):
        generator = torch.Generator(device=z.device).manual_seed(seed)
        penalty = model.lap_penalty(z, probes=probes, parents=parents, generator=generator)
        values[f"{label}_lap_penalty"] = float(penalty.detach())
    return values


def run_parent_mask_ablation(
    config: ExperimentConfig,
    output_dir: str | Path,
    device: str | torch.device | None = None,
    lambdas: Sequence[float] = (0.0, 1e4),
    resume: bool = True,
) -> pd.DataFrame:
    """Hold the memory DAG fixed and train LAP with correct vs reversed masks."""

    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else torch.device(device)
    n_patterns = max(config.pattern_counts)
    checkpoint_path = output / "parent_mask_ablation.csv"
    if resume and checkpoint_path.exists():
        rows = pd.read_csv(checkpoint_path).to_dict("records")
    else:
        rows: list[dict[str, object]] = []
    completed = {
        (int(seed), str(mask_mode), float(lambda_lap))
        for (seed, mask_mode, lambda_lap), group in pd.DataFrame(rows).groupby(
            ["seed", "mask_mode", "lambda_lap"]
        )
        if len(group) >= 11
    } if rows else set()

    def add_rows(values: Mapping[str, float], *, mask_mode: str, lambda_lap: float, seed: int, graph: str) -> None:
        for metric, value in values.items():
            rows.append({
                "mask_mode": mask_mode, "lambda_lap": lambda_lap,
                "seed": seed, "n_patterns": n_patterns, "graph": graph,
                "metric": metric, "value": value,
            })

    for seed in config.seeds:
        data_seed = seed * 10_000 + n_patterns
        chain = generate_chain(n_patterns, config.node_dim, config.p_flip, data_seed)
        fork = generate_fork(n_patterns, config.node_dim, config.p_flip, data_seed + 1)
        masks = {
            "correct": (chain.parents, fork.parents),
            "reversed": (
                reverse_parent_map(chain.nodes, chain.parents),
                reverse_parent_map(fork.nodes, fork.parents),
            ),
        }
        for mask_mode, (chain_mask, fork_mask) in masks.items():
            for lambda_lap in lambdas:
                if (seed, mask_mode, float(lambda_lap)) in completed:
                    continue
                chain_model, _ = train_energy_memory(
                    chain, lambda_lap, config.training, data_seed + 101, device, lap_parents=chain_mask
                )
                fork_model, _ = train_energy_memory(
                    fork, lambda_lap, config.training, data_seed + 202, device, lap_parents=fork_mask
                )
                chain_model, fork_model = chain_model.cpu(), fork_model.cpu()
                assert chain_model.memory_parents == dict(chain.parents)
                assert fork_model.memory_parents == dict(fork.parents)
                assert chain_model.lap_parents == {node: tuple(chain_mask[node]) for node in chain.nodes}
                assert fork_model.lap_parents == {node: tuple(fork_mask[node]) for node in fork.nodes}
                chain_retrieve = _energy_retriever(chain_model, config.retrieval)
                fork_retrieve = _energy_retriever(fork_model, config.retrieval, discretize=False)
                eval_seed = seed * 10_000_000 + n_patterns * 10_000 + int(round(config.capacity_noise * 1_000))
                retrieval = retrieval_scores(
                    chain_retrieve, chain.patterns, config.capacity_noise, seed=eval_seed
                )
                add_rows(
                    {f"chain_{key}": value for key, value in retrieval.items()},
                    mask_mode=mask_mode, lambda_lap=lambda_lap, seed=seed, graph="chain",
                )
                add_rows(
                    chain_causal_accuracy(chain_retrieve, chain),
                    mask_mode=mask_mode, lambda_lap=lambda_lap, seed=seed, graph="chain",
                )
                add_rows(
                    {f"chain_{key}": value for key, value in lap_structure_diagnostics(
                        chain_model, chain.patterns[: min(config.training.batch_size, n_patterns)],
                        chain.parents, probes=max(4, config.training.lap_probes), seed=data_seed + 303,
                    ).items()},
                    mask_mode=mask_mode, lambda_lap=lambda_lap, seed=seed, graph="chain",
                )
                add_rows(
                    {f"fork_{key}": value for key, value in fork_leakage(fork_retrieve, fork).items()},
                    mask_mode=mask_mode, lambda_lap=lambda_lap, seed=seed, graph="fork",
                )
                add_rows(
                    {f"fork_{key}": value for key, value in lap_structure_diagnostics(
                        fork_model, fork.patterns[: min(config.training.batch_size, n_patterns)],
                        fork.parents, probes=max(4, config.training.lap_probes), seed=data_seed + 404,
                    ).items()},
                    mask_mode=mask_mode, lambda_lap=lambda_lap, seed=seed, graph="fork",
                )
                pd.DataFrame(rows).to_csv(checkpoint_path, index=False)
    return pd.DataFrame(rows)


def plot_parent_mask_ablation(frame: pd.DataFrame, output_dir: str | Path) -> Path:
    """Plot whether LAP follows the supplied mask or the true causal structure."""

    output = Path(output_dir)
    panels = (
        ("fork_declared_lap_penalty", "Declared-mask penalty", True),
        ("fork_true_lap_penalty", "Ground-truth penalty", True),
        ("fork_leakage_continuous", "True intervention leakage", False),
        ("chain_exact_retrieval", "Retrieval under 10% noise", False),
    )
    fig, axes = plt.subplots(2, 2, figsize=(10.0, 7.5))
    for ax, (metric, title, log_y) in zip(axes.flat, panels):
        summary = (
            frame[frame.metric == metric].groupby(["mask_mode", "lambda_lap"]).value
            .agg(["mean", "std", "count"]).fillna(0).reset_index()
        )
        summary["ci95"] = 1.96 * summary["std"] / np.sqrt(summary["count"].clip(lower=1))
        for mode, group in summary.groupby("mask_mode"):
            means = group["mean"].clip(lower=1e-12) if log_y else group["mean"]
            if log_y:
                lower_error = np.minimum(group["ci95"], np.maximum(means - 1e-12, 0.0))
                errors = np.vstack((lower_error, group["ci95"]))
            else:
                errors = group["ci95"]
            ax.errorbar(group.lambda_lap, means, yerr=errors, marker="o", capsize=3, label=mode)
        ax.set_xscale("symlog", linthresh=1.0)
        if log_y:
            ax.set_yscale("log")
        ax.set(xlabel=r"LAP strength $\lambda$", title=title)
        ax.legend(fontsize=8)
    fig.suptitle("Correct vs reversed LAP mask (memory DAG fixed)")
    fig.tight_layout()
    path = output / "figure5_parent_mask_ablation.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return path


def run_grid(
    config: ExperimentConfig,
    output_dir: str | Path,
    device: str | torch.device | None = None,
    resume: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run baseline and LAP grids, checkpointing CSVs after every configuration."""

    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device)

    config_path = output / "config.json"
    metrics_path = output / "metrics.csv"
    history_path = output / "training_history.csv"
    serialized_config = json.dumps(
        {"schema_version": EXPERIMENT_SCHEMA_VERSION, "config": asdict(config)}, indent=2
    )
    if resume and config_path.exists() and config_path.read_text(encoding="utf-8") != serialized_config:
        raise RuntimeError("Existing checkpoint uses a different configuration; choose a new OUTPUT_DIR.")
    config_path.write_text(serialized_config, encoding="utf-8")
    environment = {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "device": str(device),
        "device_name": torch.cuda.get_device_name(device) if device.type == "cuda" else "cpu",
    }
    (output / "environment.json").write_text(
        json.dumps(environment, indent=2), encoding="utf-8"
    )
    if resume and (metrics_path.exists() != history_path.exists()):
        raise RuntimeError("Incomplete checkpoint pair: metrics.csv and training_history.csv must coexist.")
    if resume and metrics_path.exists():
        metric_rows = pd.read_csv(metrics_path).to_dict("records")
        history_rows = pd.read_csv(history_path).to_dict("records")
    else:
        metric_rows: list[dict[str, object]] = []
        history_rows: list[dict[str, object]] = []

    existing_metrics = pd.DataFrame(metric_rows)
    existing_history = pd.DataFrame(history_rows)
    completed_lap: set[tuple[int, int, float]] = set()
    if not existing_history.empty:
        terminal = existing_history[existing_history.epoch == config.training.epochs]
        for (seed, n_patterns, lambda_lap), group in terminal.groupby(
            ["seed", "n_patterns", "lambda_lap"]
        ):
            if group.graph.nunique() == 2:
                completed_lap.add((int(seed), int(n_patterns), float(lambda_lap)))

    def baseline_complete(model: str, seed: int, n_patterns: int) -> bool:
        if existing_metrics.empty:
            return False
        subset = existing_metrics[
            (existing_metrics.model == model)
            & (existing_metrics.seed == seed)
            & (existing_metrics.n_patterns == n_patterns)
        ]
        return {"chain", "fork"}.issubset(set(subset.graph))

    for seed in config.seeds:
        for n_patterns in config.pattern_counts:
            # The seed depends on N but not lambda: all lambdas see identical data.
            data_seed = seed * 10_000 + n_patterns
            chain = generate_chain(n_patterns, config.node_dim, config.p_flip, data_seed)
            fork = generate_fork(n_patterns, config.node_dim, config.p_flip, data_seed + 1)

            baseline_chain = ModernHopfield(chain.patterns)
            baseline_fork = ModernHopfield(fork.patterns)
            if not baseline_complete("Modern Hopfield", seed, n_patterns):
                metric_rows.extend(
                    _evaluate_model(
                        _hopfield_retriever(baseline_chain),
                        chain,
                        fork,
                        config,
                        model_name="Modern Hopfield",
                        lambda_lap=None,
                        seed=seed,
                    )
                )
                metric_rows.extend(
                    _metric_rows(
                    fork_leakage(_hopfield_retriever(baseline_fork, discretize=False), fork),
                    model="Modern Hopfield",
                    lambda_lap=None,
                    n_patterns=n_patterns,
                    seed=seed,
                    graph="fork",
                    evaluation="intervention_leakage",
                    )
                )

            # Architectural oracle: identical local memories with the residual path disabled.
            hard_chain = CausalEnergyMemory(
                chain.nodes, chain.parents, chain.patterns, chain.node_dim,
                hidden_dim=config.training.hidden_dim, residual_scale=0.0,
                hopfield_beta=config.training.hopfield_beta,
            )
            hard_fork = CausalEnergyMemory(
                fork.nodes, fork.parents, fork.patterns, fork.node_dim,
                hidden_dim=config.training.hidden_dim, residual_scale=0.0,
                hopfield_beta=config.training.hopfield_beta,
            )
            if not baseline_complete("Hard-mask local Hopfield", seed, n_patterns):
                metric_rows.extend(
                    _evaluate_model(
                        _energy_retriever(hard_chain, config.retrieval),
                        chain, fork, config, model_name="Hard-mask local Hopfield",
                        lambda_lap=None, seed=seed,
                    )
                )
                metric_rows.extend(
                    _metric_rows(
                    fork_leakage(
                        _energy_retriever(hard_fork, config.retrieval, discretize=False), fork
                    ),
                    model="Hard-mask local Hopfield", lambda_lap=None,
                    n_patterns=n_patterns, seed=seed, graph="fork",
                    evaluation="intervention_leakage",
                    )
                )

            for lambda_lap in config.lambdas:
                if (seed, n_patterns, float(lambda_lap)) in completed_lap:
                    continue
                chain_model, chain_history = train_energy_memory(
                    chain, lambda_lap, config.training, data_seed + 101, device
                )
                fork_model, fork_history = train_energy_memory(
                    fork, lambda_lap, config.training, data_seed + 202, device
                )
                chain_model = chain_model.cpu()
                fork_model = fork_model.cpu()
                metric_rows.extend(
                    _evaluate_model(
                        _energy_retriever(chain_model, config.retrieval),
                        chain,
                        fork,
                        config,
                        model_name="LAP E-SCM",
                        lambda_lap=lambda_lap,
                        seed=seed,
                    )
                )
                metric_rows.extend(
                    _metric_rows(
                        fork_leakage(
                            _energy_retriever(fork_model, config.retrieval, discretize=False), fork
                        ),
                        model="LAP E-SCM",
                        lambda_lap=lambda_lap,
                        n_patterns=n_patterns,
                        seed=seed,
                        graph="fork",
                        evaluation="intervention_leakage",
                    )
                )
                for graph, frame in (("chain", chain_history), ("fork", fork_history)):
                    records = frame.assign(
                        graph=graph,
                        lambda_lap=lambda_lap,
                        n_patterns=n_patterns,
                        seed=seed,
                    ).to_dict("records")
                    history_rows.extend(records)

                pd.DataFrame(metric_rows).to_csv(metrics_path, index=False)
                pd.DataFrame(history_rows).to_csv(history_path, index=False)

    pd.DataFrame(metric_rows).to_csv(metrics_path, index=False)
    pd.DataFrame(history_rows).to_csv(history_path, index=False)
    return pd.DataFrame(metric_rows), pd.DataFrame(history_rows)


def _capacity_by_lambda(metrics: pd.DataFrame, threshold: float = 0.95) -> pd.DataFrame:
    subset = metrics[
        (metrics.metric == "exact_retrieval")
        & (metrics.evaluation == "capacity")
    ].copy()
    curve = (
        subset.groupby(["model", "lambda_lap", "seed", "n_patterns"], dropna=False, as_index=False)
        .value.mean()
    )
    rows: list[dict[str, object]] = []
    for keys, group in curve.groupby(["model", "lambda_lap", "seed"], dropna=False):
        valid = group[group.value >= threshold]
        rows.append(
            {
                "model": keys[0],
                "lambda_lap": keys[1],
                "seed": keys[2],
                "capacity": 0 if valid.empty else int(valid.n_patterns.max()),
            }
        )
    return pd.DataFrame(rows)


def _bootstrap_mean_ci(
    values: Sequence[float], *, bootstrap_samples: int = 10_000, seed: int = 0
) -> tuple[float, float, float]:
    array = np.asarray(values, dtype=float)
    mean = float(array.mean())
    if len(array) < 2:
        return mean, float("nan"), float("nan")
    generator = np.random.default_rng(seed)
    boot = generator.choice(array, size=(bootstrap_samples, len(array)), replace=True).mean(axis=1)
    low, high = np.quantile(boot, [0.025, 0.975])
    return mean, float(low), float(high)


def paired_leakage_effects(metrics: pd.DataFrame) -> pd.DataFrame:
    """Paired effect, bootstrap CI, and exact sign-flip test versus lambda=0."""

    per_seed = (
        metrics[(metrics.model == "LAP E-SCM") & (metrics.metric == "leakage_continuous")]
        .groupby(["seed", "lambda_lap"], as_index=False).value.mean()
    )
    baseline = per_seed[per_seed.lambda_lap == 0][["seed", "value"]].rename(columns={"value": "baseline"})
    retrieval = (
        metrics[
            (metrics.model == "LAP E-SCM")
            & (metrics.metric == "exact_retrieval")
            & (metrics.evaluation == "capacity")
        ].groupby(["seed", "lambda_lap"], as_index=False).value.mean()
    )
    retrieval_baseline = retrieval[retrieval.lambda_lap == 0][["seed", "value"]].rename(
        columns={"value": "retrieval_baseline"}
    )
    rows: list[dict[str, float | int]] = []
    for lambda_lap, group in per_seed.groupby("lambda_lap"):
        paired = group.merge(baseline, on="seed", validate="one_to_one")
        absolute_reduction = paired.baseline - paired.value
        relative_reduction = absolute_reduction / paired.baseline.clip(lower=1e-12)
        abs_mean, abs_low, abs_high = _bootstrap_mean_ci(absolute_reduction, seed=int(lambda_lap) + 17)
        rel_mean, rel_low, rel_high = _bootstrap_mean_ci(relative_reduction, seed=int(lambda_lap) + 29)
        signs = np.asarray(list(itertools.product((-1.0, 1.0), repeat=len(absolute_reduction))))
        null_means = (signs * absolute_reduction.to_numpy()).mean(axis=1)
        p_one_sided = float((null_means >= abs_mean - 1e-15).mean())
        retrieval_paired = retrieval[retrieval.lambda_lap == lambda_lap].merge(
            retrieval_baseline, on="seed", validate="one_to_one"
        )
        retrieval_delta = retrieval_paired.value - retrieval_paired.retrieval_baseline
        retrieval_mean, retrieval_low, retrieval_high = _bootstrap_mean_ci(
            retrieval_delta, seed=int(lambda_lap) + 41
        )
        rows.append({
            "lambda_lap": float(lambda_lap), "n_seeds": len(paired),
            "absolute_reduction_mean": abs_mean, "absolute_ci95_low": abs_low, "absolute_ci95_high": abs_high,
            "relative_reduction_mean": rel_mean, "relative_ci95_low": rel_low, "relative_ci95_high": rel_high,
            "retrieval_delta_mean": retrieval_mean, "retrieval_delta_ci95_low": retrieval_low, "retrieval_delta_ci95_high": retrieval_high,
            "p_sign_flip_one_sided": p_one_sided,
        })
    result = pd.DataFrame(rows)
    tested = result.index[result.lambda_lap > 0].to_numpy()
    if len(tested):
        order = tested[np.argsort(result.loc[tested, "p_sign_flip_one_sided"].to_numpy())]
        adjusted: dict[int, float] = {}
        running = 0.0
        for rank, index in enumerate(order):
            candidate = (len(order) - rank) * result.loc[index, "p_sign_flip_one_sided"]
            running = max(running, float(candidate))
            adjusted[int(index)] = min(1.0, running)
        result["p_holm"] = np.nan
        for index, value in adjusted.items():
            result.loc[index, "p_holm"] = value
    return result


def paper_claim_gate(
    metrics: pd.DataFrame,
    paired_effects: pd.DataFrame,
    ablation_metrics: pd.DataFrame,
    config: ExperimentConfig,
    *,
    selected_lambda: float = 1e4,
    retrieval_noninferiority_margin: float = 0.05,
) -> pd.DataFrame:
    """Predeclared gates for the narrow claim that LAP suppresses supplied-mask leakage."""

    effect = paired_effects.loc[paired_effects.lambda_lap == selected_lambda]
    if len(effect) != 1:
        raise ValueError(f"selected lambda {selected_lambda:g} is missing or duplicated")
    effect = effect.iloc[0]
    mask_leakage = ablation_metrics[
        (ablation_metrics.lambda_lap == selected_lambda)
        & (ablation_metrics.metric == "fork_leakage_continuous")
    ].pivot_table(index="seed", columns="mask_mode", values="value", aggfunc="mean")
    if {"correct", "reversed"}.issubset(mask_leakage.columns):
        mask_difference = mask_leakage.reversed - mask_leakage.correct
        mask_mean, mask_low, mask_high = _bootstrap_mean_ci(mask_difference, seed=71)
    else:
        mask_mean = mask_low = mask_high = float("nan")
    hard_leakage = metrics[
        (metrics.model == "Hard-mask local Hopfield")
        & (metrics.metric == "leakage_continuous")
    ].value.mean()
    checks = [
        ("independent_seeds", len(config.seeds), ">= 10", len(config.seeds) >= 10),
        ("leakage_reduction_ci", effect.absolute_ci95_low, "> 0", effect.absolute_ci95_low > 0),
        ("leakage_reduction_holm", effect.p_holm, "< 0.05", effect.p_holm < 0.05),
        ("retrieval_noninferiority", effect.retrieval_delta_ci95_low, f"> {-retrieval_noninferiority_margin:g}", effect.retrieval_delta_ci95_low > -retrieval_noninferiority_margin),
        ("correct_beats_reversed_mask", mask_low, "> 0", mask_low > 0),
        ("hard_mask_leakage_oracle", hard_leakage, "< 1e-6", hard_leakage < 1e-6),
    ]
    result = pd.DataFrame(checks, columns=["criterion", "evidence", "threshold", "passed"])
    result.attrs["mask_difference_mean"] = mask_mean
    result.attrs["mask_difference_ci95"] = (mask_low, mask_high)
    return result


def plot_results(metrics: pd.DataFrame, history: pd.DataFrame, output_dir: str | Path) -> list[Path]:
    """Create the four primary experiment figures."""

    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    paths: list[Path] = []
    plt.style.use("seaborn-v0_8-whitegrid")

    lap_metrics = metrics[metrics.model == "LAP E-SCM"]
    capacities = _capacity_by_lambda(lap_metrics)
    leakage_by_seed = (
        lap_metrics[lap_metrics.metric == "leakage_continuous"]
        .groupby(["lambda_lap", "seed"], as_index=False).value.mean()
    )
    leakage = leakage_by_seed.groupby("lambda_lap").value.agg(["mean", "std", "count"]).fillna(0).reset_index()
    leakage["ci95"] = 1.96 * leakage["std"] / np.sqrt(leakage["count"].clip(lower=1))
    capacity_summary = (
        capacities.groupby("lambda_lap").capacity.agg(["mean", "std", "count"]).fillna(0).reset_index()
    )
    capacity_summary["ci95"] = 1.96 * capacity_summary["std"] / np.sqrt(capacity_summary["count"].clip(lower=1))
    fig, ax1 = plt.subplots(figsize=(7.2, 4.5))
    ax1.errorbar(leakage.lambda_lap, leakage["mean"], yerr=leakage["ci95"], fmt="o-", capsize=3, color="#c44e52", label="Leakage")
    ax1.set_xscale("symlog", linthresh=1.0)
    ax1.set_xlabel(r"LAP strength $\lambda$")
    ax1.set_ylabel("Continuous normalized intervention leakage", color="#c44e52")
    ax2 = ax1.twinx()
    ax2.errorbar(capacity_summary.lambda_lap, capacity_summary["mean"], yerr=capacity_summary["ci95"], fmt="s--", capsize=3, color="#4c72b0", label="Capacity")
    ax2.set_ylabel("Largest tested N with ≥95% retrieval", color="#4c72b0")
    fig.suptitle("LAP modularity–capacity trade-off")
    fig.tight_layout()
    path = output / "figure1_tradeoff.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    paths.append(path)

    capacity_curve = metrics[
        (metrics.metric == "exact_retrieval") & (metrics.evaluation == "capacity")
    ].groupby(["model", "lambda_lap", "n_patterns"], dropna=False).value.agg(["mean", "std", "count"]).fillna(0).reset_index()
    capacity_curve["ci95"] = 1.96 * capacity_curve["std"] / np.sqrt(capacity_curve["count"].clip(lower=1))
    lap_values = sorted(float(value) for value in lap_metrics.lambda_lap.dropna().unique())
    lap_styles = {
        value: (2.0 + 2.0 * (len(lap_values) - 1 - index), 6.0 + 2.0 * (len(lap_values) - 1 - index))
        for index, value in enumerate(lap_values)
    }

    def curve_style(model: str, lambda_lap: float) -> tuple[str, float, float, str]:
        if model == "LAP E-SCM":
            linewidth, markersize = lap_styles[float(lambda_lap)]
            return f"LAP λ={lambda_lap:g}", linewidth, markersize, "-"
        if model == "Hard-mask local Hopfield":
            return model, 2.5, 6.0, "--"
        return model, 2.0, 6.0, "-"

    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    for (model, lam), group in capacity_curve.groupby(["model", "lambda_lap"], dropna=False):
        label, linewidth, markersize, linestyle = curve_style(model, lam)
        ax.errorbar(group.n_patterns, group["mean"], yerr=group["ci95"], marker="o", capsize=3, linewidth=linewidth, markersize=markersize, linestyle=linestyle, label=label)
    ax.axhline(0.95, color="black", linestyle=":", linewidth=1, label="95% threshold")
    ax.set(xlabel="Stored patterns N", ylabel="Exact retrieval rate", ylim=(-0.03, 1.03), title="Capacity curves")
    ax.legend(fontsize=8)
    fig.tight_layout()
    path = output / "figure2_capacity.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    paths.append(path)

    max_n = int(metrics.n_patterns.max())
    basin = metrics[
        (metrics.metric == "exact_retrieval")
        & (metrics.n_patterns == max_n)
        & (metrics.evaluation == "basin")
    ].groupby(["model", "lambda_lap", "noise"], dropna=False).value.agg(["mean", "std", "count"]).fillna(0).reset_index()
    basin["ci95"] = 1.96 * basin["std"] / np.sqrt(basin["count"].clip(lower=1))
    fig, ax = plt.subplots(figsize=(7.2, 4.5))
    for (model, lam), group in basin.groupby(["model", "lambda_lap"], dropna=False):
        label, linewidth, markersize, linestyle = curve_style(model, lam)
        ax.errorbar(100 * group.noise, group["mean"], yerr=group["ci95"], marker="o", capsize=3, linewidth=linewidth, markersize=markersize, linestyle=linestyle, label=label)
    ax.set(xlabel="Initial bit-flip noise (%)", ylabel="Exact retrieval rate", ylim=(-0.03, 1.03), title=f"Basin of attraction (N={max_n})")
    ax.legend(fontsize=8)
    fig.tight_layout()
    path = output / "figure3_basin.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    paths.append(path)

    diagnostic_source = history.copy()
    diagnostic_source["lap_to_task_ratio"] = diagnostic_source.weighted_lap_loss / diagnostic_source.task_loss.clip(lower=1e-12)
    diagnostic_by_seed = (
        diagnostic_source.groupby(["lambda_lap", "epoch", "seed"], as_index=False)
        [["lap_loss", "lap_to_task_ratio"]].mean()
    )
    diagnostic = (
        diagnostic_by_seed.groupby(["lambda_lap", "epoch"])
        .agg(lap_mean=("lap_loss", "mean"), lap_std=("lap_loss", "std"), ratio_mean=("lap_to_task_ratio", "mean"), ratio_std=("lap_to_task_ratio", "std"), count=("seed", "nunique"))
        .fillna(0).reset_index()
    )
    diagnostic["lap_ci95"] = 1.96 * diagnostic.lap_std / np.sqrt(diagnostic["count"].clip(lower=1))
    diagnostic["ratio_ci95"] = 1.96 * diagnostic.ratio_std / np.sqrt(diagnostic["count"].clip(lower=1))
    fig, (ax, ax_ratio) = plt.subplots(1, 2, figsize=(11.0, 4.5))
    for lam, group in diagnostic.groupby("lambda_lap"):
        lap_mean = group.lap_mean.clip(lower=1e-12)
        ax.plot(group.epoch, lap_mean, label=f"λ={lam:g}")
        ax.fill_between(group.epoch, (lap_mean - group.lap_ci95).clip(lower=1e-12), lap_mean + group.lap_ci95, alpha=0.12)
        ax_ratio.plot(group.epoch, group.ratio_mean, label=f"λ={lam:g}")
        ax_ratio.fill_between(group.epoch, (group.ratio_mean - group.ratio_ci95).clip(lower=0), group.ratio_mean + group.ratio_ci95, alpha=0.12)
    ax.set_yscale("log")
    ax.set(xlabel="Epoch", ylabel="Raw LAP mixed-partial penalty", title="Illegal coupling")
    ax_ratio.set_yscale("symlog", linthresh=1e-4)
    ax_ratio.set(xlabel="Epoch", ylabel=r"$\lambda L_{LAP}/L_{task}$", title="Regularizer scale")
    ax.legend(fontsize=8)
    ax_ratio.legend(fontsize=8)
    fig.tight_layout()
    path = output / "figure4_lap_training.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    paths.append(path)
    return paths


## 2. 选择实验规模

先用 `smoke` 排除代码错误，再用 `confirmatory` 检查五个 seed 下效应是否稳定，最后才运行 `full` 的十个 seed 完整容量扫描。论文结论只能依据 `full`；三种配置都扫描 $\lambda\in\{0,10^2,10^3,10^4,10^5\}$。

In [ ]:
from pathlib import Path
import torch

EXPERIMENT_MODE = "smoke"  # one of: smoke, confirmatory, full
configs = {
    "smoke": ExperimentConfig(),
    "confirmatory": ExperimentConfig.confirmatory(),
    "full": ExperimentConfig.full(),
}
if EXPERIMENT_MODE not in configs:
    raise ValueError(f"Unknown EXPERIMENT_MODE: {EXPERIMENT_MODE}")
config = configs[EXPERIMENT_MODE]
OUTPUT_DIR = Path(f"/content/lap_results/{EXPERIMENT_MODE}_v{EXPERIMENT_SCHEMA_VERSION}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(config)
print("device =", device, "| output =", OUTPUT_DIR)

## 3. 先做吸引子诊断（不要先解释最终曲线）

先单独训练 $\lambda=0$ 的最小 E-SCM，并检查三个必要条件：存储点的梯度范数应接近 0、Hessian 最小特征值应为正、0% 噪声检索率应接近 1。任一条件失败，都先停止解释 LAP trade-off。

In [ ]:
#@title 训练 λ=0 的最小模型并测量 stationarity / curvature

def attractor_diagnostics(model, patterns, max_patterns=3):
    """Measure whether stored patterns are stationary local minima."""
    model.eval()
    rows = []
    for index in range(min(max_patterns, len(patterns))):
        point = patterns[index].detach().clone().requires_grad_(True)

        def scalar_energy(z):
            return model(z.unsqueeze(0)).sum()

        energy = scalar_energy(point)
        gradient = torch.autograd.grad(energy, point)[0]
        hessian = torch.autograd.functional.hessian(scalar_energy, point)
        hessian = 0.5 * (hessian + hessian.T)
        min_eigenvalue = torch.linalg.eigvalsh(hessian).min()
        rows.append({
            "pattern": index,
            "energy": float(energy.detach()),
            "gradient_norm": float(gradient.norm().detach()),
            "min_hessian_eigenvalue": float(min_eigenvalue.detach()),
        })
    return pd.DataFrame(rows)

# Use the same data/training seeds as the corresponding quick-grid configuration.
diagnostic_n = config.pattern_counts[-1]
diagnostic_data_seed = diagnostic_n
diagnostic_data = generate_chain(
    diagnostic_n,
    node_dim=config.node_dim,
    p_flip=config.p_flip,
    seed=diagnostic_data_seed,
)
diagnostic_model, diagnostic_history = train_energy_memory(
    diagnostic_data,
    lambda_lap=0.0,
    config=config.training,
    seed=diagnostic_data_seed + 101,
    device=torch.device(device),
)
diagnostic_patterns = diagnostic_data.patterns.to(device)
diagnostics = attractor_diagnostics(diagnostic_model, diagnostic_patterns)
zero_noise = retrieval_scores(
    _energy_retriever(diagnostic_model, config.retrieval),
    diagnostic_patterns,
    noise=0.0,
)

display(diagnostics)
print("0% noise exact retrieval:", zero_noise["exact_retrieval"])
print("判据：gradient_norm ≈ 0，min_hessian_eigenvalue > 0，exact retrieval ≈ 1。")
if (
    diagnostics.gradient_norm.max() > 1e-3
    or diagnostics.min_hessian_eigenvalue.min() <= 0
    or zero_noise["exact_retrieval"] < 0.95
):
    print("诊断失败：存储 pattern 尚未形成稳定吸引子；不要解释后续 LAP 权衡图。")
else:
    print("诊断通过：可以继续评估 LAP 泄漏与容量权衡。")


如果要防止 Colab 断连丢失结果，可先挂载 Drive，然后把 `OUTPUT_DIR` 改成 `/content/drive/MyDrive/lap_results/...`。

In [ ]:
# 可选：保存到 Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_DIR = Path(f'/content/drive/MyDrive/lap_results/{EXPERIMENT_MODE}_v{EXPERIMENT_SCHEMA_VERSION}')

## 4. 训练、评估与 checkpoint

In [ ]:
metrics, history = run_grid(config, OUTPUT_DIR, device=device)
figure_paths = plot_results(metrics, history, OUTPUT_DIR)
paired_effects = paired_leakage_effects(metrics)
paired_effects.to_csv(OUTPUT_DIR / "paired_leakage_effects.csv", index=False)
ablation_metrics = run_parent_mask_ablation(config, OUTPUT_DIR, device=device)
figure_paths.append(plot_parent_mask_ablation(ablation_metrics, OUTPUT_DIR))
claim_gate = paper_claim_gate(metrics, paired_effects, ablation_metrics, config)
claim_gate.to_csv(OUTPUT_DIR / "paper_claim_gate.csv", index=False)
print(f"完成：{len(metrics)} 条主指标，{len(history)} 条训练记录，{len(ablation_metrics)} 条 mask 消融指标")
display(metrics.head(12))

## 5. 核心汇总

In [ ]:
summary = (metrics
           .groupby(['model', 'lambda_lap', 'evaluation', 'metric'], dropna=False)['value']
           .agg(['mean', 'std', 'count'])
           .reset_index())
display(summary)
display(paired_effects)
display(claim_gate)

loss_scale_summary = (history
                      .groupby('lambda_lap')[['task_loss', 'lap_loss', 'weighted_lap_loss']]
                      .agg(['mean', 'std']))
display(loss_scale_summary)

ablation_summary = (ablation_metrics
                    .groupby(['mask_mode', 'lambda_lap', 'metric'])['value']
                    .agg(['mean', 'std', 'count'])
                    .reset_index())
display(ablation_summary)

## 6. 五张实验图

In [ ]:
from IPython.display import Image, display
for path in figure_paths:
    print(path.name)
    display(Image(filename=str(path)))

## 读图边界

图中误差棒/阴影是跨独立 seed 的近似 95% CI；效应表另给配对 bootstrap CI。图 1 以 matched-counterfactual 的连续泄漏为主指标：factual-control 与 do(X) 两支都释放 Y 松弛，再比较终态，`leakage_bit_fraction` 只作补充。先确认图 4 左侧 raw LAP penalty 随 $\lambda$ 下降，并检查右侧 $\lambda L_{LAP}/L_{task}$ 已进入有效量级，再解释泄漏或容量差异。图 5 必须同时查看 declared-mask 与 ground-truth penalty：若 reversed mask 只压低前者而不能改善真实泄漏，就证明 LAP 在执行外部提供的结构约束，而非发现因果方向。hard-mask local Hopfield 是架构上界，不是可学习因果基线。烟雾与 confirmatory 配置只验证流水线和效应稳定性，正式结论必须使用 `full` 的十 seed 完整配置、误差棒、配对置信区间和 Holm 校正检验。